# WordPiece Tokenizer

This notebook explains the **WordPiece** tokenization algorithm step by step. WordPiece is the subword tokenization method used by BERT and several BERT-style encoder models.

The goal of this lecture is to make the algorithm visible. We will train a tiny WordPiece tokenizer from scratch, inspect the scores it uses to choose merges, and then use the learned vocabulary to tokenize new words with the longest-match rule used during inference.

Reference: Hugging Face LLM Course, Chapter 6, section 6, "WordPiece tokenization": https://huggingface.co/learn/llm-course/chapter6/6

## Learning Objectives

By the end of this notebook, you should be able to:

- Explain why language models use **subword tokens** instead of only words or characters.
- Describe the WordPiece training idea: start with characters, score adjacent pairs, merge the highest-scoring pair, and repeat.
- Explain the meaning of continuation tokens such as `##ing`.
- Apply the WordPiece longest-match tokenization algorithm to a new word or sentence.
- Compare WordPiece with BPE and identify why they may learn different merge rules from the same corpus.

## 1. Why Tokenization Matters

A neural language model does not directly read a Python string such as:

```text
"tokenization is useful"
```

It reads a sequence of integer token IDs:

```text
[19204, 2003, 6179]
```

A **tokenizer** is the component that converts text into tokens and token IDs. The model then looks up each token ID in an embedding table.

There are three simple ways to tokenize text:

| Approach | Example for `playing` | Strength | Weakness |
| --- | --- | --- | --- |
| Word-level | `playing` | Easy to interpret | Huge vocabulary; unknown words are common |
| Character-level | `p l a y i n g` | Can represent any word | Long sequences; harder for the model to learn larger units |
| Subword-level | `play ##ing` | Balanced vocabulary and sequence length | Tokens are less intuitive than words |

WordPiece is a **subword tokenization** algorithm. It learns common pieces from data instead of relying on hand-written rules.

## 2. The Main Idea of WordPiece

WordPiece starts with small tokens, usually characters.

For words after pre-tokenization, it marks characters that are **inside** a word with a continuation prefix, usually `##`:

```text
hug  -> h ##u ##g
pug  -> p ##u ##g
hugs -> h ##u ##g ##s
```

The prefix means: this token is not allowed to begin a new word by itself. For example, `##ing` means "the letters `ing` attached to something before them."

Like BPE, WordPiece repeatedly merges adjacent tokens. The difference is the selection rule. BPE chooses the most frequent adjacent pair. WordPiece chooses the pair with the highest score:

```text
score(pair) = frequency(pair) / (frequency(first_token) * frequency(second_token))
```

This score favors pairs whose two parts occur together more often than they occur separately.

## 3. WordPiece Training Algorithm

A WordPiece tokenizer is usually trained after two earlier steps:

1. **Normalization**: optional text cleanup, such as lowercasing, accent stripping, or Unicode normalization.
2. **Pre-tokenization**: splitting text into rough units, often words and punctuation.

### Where pre-tokenization fits

The full path from text to model input is:

```text
raw text → normalization → pre-tokenization → WordPiece inside each word → token IDs
```

**Pre-tokenization** splits text into rough words *before* WordPiece runs. WordPiece then trains and tokenizes **inside each word only**. Its output does three jobs:

1. **It defines the words that `##` refers to.** The first piece of each pre-tokenized word has no prefix, and every later piece gets `##`. The pre-tokenizer therefore decides where `##` can appear. In `"tokenization."`, the `.` is its own word, so it is tokenized as `.` and never as `##.`.
2. **It makes training tractable.** Training only needs each unique word and its count. If `"This"` appears 3 times, it is processed once with weight 3, which is why Section 4 starts from `word_freqs` rather than raw text.
3. **It discards spaces.** BERT-style pre-tokenizers split on whitespace and throw the spaces away. This is the opposite of GPT-2's byte-level BPE, which keeps a space attached to the following word (`Ġis`). As a result, `"tokenization."` and `"tokenization ."` produce the same tokens, and a decoder cannot tell which one the original text was.

BERT's real pre-tokenizer splits on whitespace and makes **every** punctuation character, including the apostrophe, its own word:

```text
"Hello, I'm learning tokenization in 2024!"
→ ['Hello', ',', 'I', "'", 'm', 'learning', 'tokenization', 'in', '2024', '!']
```

For uncased BERT models, a normalizer runs first. It lowercases the text and removes accents, so `"Héllo Café"` becomes `"hello cafe"`. Section 8 builds a simpler pre-tokenizer that separates only `. , ! ? ; :` and does no normalization.

### The training loop

For a classroom version of WordPiece, the training loop is:

1. Count how often each word appears in the corpus.
2. Create the initial vocabulary from characters. The first character of a word is stored normally, and later characters receive the `##` prefix.
3. Split each word into these initial WordPiece symbols.
4. Count token frequencies and adjacent pair frequencies, weighted by word frequency.
5. Compute a score for every adjacent pair.
6. Merge the highest-scoring pair everywhere it appears.
7. Add the merged token to the vocabulary.
8. Repeat until the vocabulary reaches the desired size or the desired number of merges is learned.

The important point: **WordPiece is still greedy**. At each step, it chooses the best pair according to its current score.

## 4. A Tiny Worked Example

We will use the same toy vocabulary as the BPE notebook and the Hugging Face lesson:

```text
("hug", 10), ("pug", 5), ("pun", 12), ("bun", 4), ("hugs", 5)
```

The numbers are word frequencies. For example, `hug` appears 10 times and `pun` appears 12 times.

The initial WordPiece tokens are:

```text
hug  -> h ##u ##g
pug  -> p ##u ##g
pun  -> p ##u ##n
bun  -> b ##u ##n
hugs -> h ##u ##g ##s
```

The initial vocabulary is:

```text
[UNK], b, h, p, ##g, ##n, ##s, ##u
```

We include `[UNK]` because WordPiece tokenizers need a fallback token for words that cannot be segmented using the vocabulary.

In [2]:
from collections import Counter, defaultdict

UNK_TOKEN = "[UNK]"
CONTINUING_SUBWORD_PREFIX = "##"

word_freqs = {
    "hug": 10,
    "pug": 5,
    "pun": 12,
    "bun": 4,
    "hugs": 5,
}

def initial_wordpiece_split(word):
    """Split a word into WordPiece-style starting characters."""
    if not word:
        return []
    return [word[0]] + [CONTINUING_SUBWORD_PREFIX + char for char in word[1:]]

def initial_wordpiece_vocab(word_freqs, unk_token=UNK_TOKEN):
    """Create the initial WordPiece vocabulary from word characters."""
    vocab = {unk_token}
    for word in word_freqs:
        if not word:
            continue
        vocab.add(word[0])
        for char in word[1:]:
            vocab.add(CONTINUING_SUBWORD_PREFIX + char)
    return sorted(vocab)

splits = {word: initial_wordpiece_split(word) for word in word_freqs}
base_vocab = initial_wordpiece_vocab(word_freqs)

print("Initial vocabulary:", base_vocab)
print("Initial splits:")
for word, split in splits.items():
    print(f"  {word:4s} -> {split}  frequency={word_freqs[word]}")

Initial vocabulary: ['##g', '##n', '##s', '##u', '[UNK]', 'b', 'h', 'p']
Initial splits:
  hug  -> ['h', '##u', '##g']  frequency=10
  pug  -> ['p', '##u', '##g']  frequency=5
  pun  -> ['p', '##u', '##n']  frequency=12
  bun  -> ['b', '##u', '##n']  frequency=4
  hugs -> ['h', '##u', '##g', '##s']  frequency=5


### Counting Token and Pair Frequencies

WordPiece needs two kinds of counts.

For `hug` with frequency 10, the tokens are:

```text
h, ##u, ##g
```

So each token receives `+10`. The adjacent pairs are:

```text
(h, ##u) and (##u, ##g)
```

So both pairs also receive `+10`.

For `hugs` with frequency 5, the pairs are:

```text
(h, ##u), (##u, ##g), and (##g, ##s)
```

So each receives `+5`.

In [3]:
def compute_wordpiece_stats(splits, word_freqs):
    """Count token frequencies and adjacent pair frequencies."""
    token_freqs = defaultdict(int)
    pair_freqs = defaultdict(int)

    for word, freq in word_freqs.items():
        split = splits[word]
        for token in split:
            token_freqs[token] += freq
        for i in range(len(split) - 1):
            pair = (split[i], split[i + 1])
            pair_freqs[pair] += freq

    return token_freqs, pair_freqs

def compute_pair_scores(splits, word_freqs):
    """Compute the WordPiece score for every adjacent pair."""
    token_freqs, pair_freqs = compute_wordpiece_stats(splits, word_freqs)
    scores = {}

    for pair, pair_freq in pair_freqs.items():
        first, second = pair
        scores[pair] = pair_freq / (token_freqs[first] * token_freqs[second])

    return scores, token_freqs, pair_freqs

scores, token_freqs, pair_freqs = compute_pair_scores(splits, word_freqs)

print("Token frequencies:")
for token, freq in sorted(token_freqs.items()):
    print(f"  {token:4s}: {freq}")

print("\nPair scores before the first merge:")
for pair, score in sorted(scores.items(), key=lambda item: (-item[1], -pair_freqs[item[0]], item[0])):
    print(f"  {pair}: pair_freq={pair_freqs[pair]:2d}, score={score:.4f}")

Token frequencies:
  ##g : 20
  ##n : 16
  ##s : 5
  ##u : 36
  b   : 4
  h   : 15
  p   : 17

Pair scores before the first merge:
  ('##g', '##s'): pair_freq= 5, score=0.0500
  ('##u', '##g'): pair_freq=20, score=0.0278
  ('p', '##u'): pair_freq=17, score=0.0278
  ('##u', '##n'): pair_freq=16, score=0.0278
  ('h', '##u'): pair_freq=15, score=0.0278
  ('b', '##u'): pair_freq= 4, score=0.0278


The highest-scoring pair is merged first.

In this example, `("##g", "##s")` has a higher score than `("##u", "##g")`, even though it appears fewer times.

Why? `##g` and `##s` occur together very consistently in this tiny corpus, while `##u` appears with several different neighbors. WordPiece therefore treats `##g ##s` as a strong subword candidate:

```text
("##g", "##s") -> "##gs"
```

This is the main contrast with BPE: raw frequency is not the whole story.

In [4]:
def merge_wordpiece_pair(pair, splits):
    """Merge one WordPiece pair everywhere it appears in every split word."""
    first, second = pair
    merged_token = first + second.removeprefix(CONTINUING_SUBWORD_PREFIX)
    new_splits = {}

    for word, split in splits.items():
        merged = []
        i = 0
        while i < len(split):
            if i < len(split) - 1 and split[i] == first and split[i + 1] == second:
                merged.append(merged_token)
                i += 2
            else:
                merged.append(split[i])
                i += 1
        new_splits[word] = merged

    return new_splits, merged_token

def select_best_pair(scores, pair_freqs):
    """Select the highest-scoring pair with deterministic tie breaking."""
    return max(scores, key=lambda pair: (scores[pair], pair_freqs[pair], pair))

best_pair = select_best_pair(scores, pair_freqs)
print("Best pair:", best_pair)
print("Pair frequency:", pair_freqs[best_pair])
print("WordPiece score:", scores[best_pair])

splits_after_first_merge, first_merged_token = merge_wordpiece_pair(best_pair, splits)

print("\nSplits after merging", best_pair, "->", first_merged_token)
for word, split in splits_after_first_merge.items():
    print(f"  {word:4s} -> {split}")

Best pair: ('##g', '##s')
Pair frequency: 5
WordPiece score: 0.05

Splits after merging ('##g', '##s') -> ##gs
  hug  -> ['h', '##u', '##g']
  pug  -> ['p', '##u', '##g']
  pun  -> ['p', '##u', '##n']
  bun  -> ['b', '##u', '##n']
  hugs -> ['h', '##u', '##gs']


### Visual walkthrough: count → score → merge → recount

Run the next cell after the scoring and merging functions above. Click **Next merge** to follow five training steps; **Reset** returns to the first step. Each view shows the current splits, the WordPiece score of every adjacent pair, and the result of merging the winning pair everywhere it occurs. Highlighted tiles mark the selected pair before merging and the new token afterward.

The bar lengths represent **scores**, not pair counts. Each bar's label shows the full calculation, `pair count / (count of first token × count of second token)`, using the token counts listed above the bars. For the first step, `score(##g, ##s) = 5 / (20 × 5) = 0.05`, which beats `score(##u, ##g) = 20 / (36 × 20) ≈ 0.0278` even though `(##u, ##g)` is four times more frequent. After a merge, both the token counts and the pair counts change, so every score is recomputed.

Compare this with the BPE walkthrough, which ranks pairs by count alone. The two algorithms start from the same words but select different merges.

The last line applies the current vocabulary to an unseen word, `bugs`, using the longest-match search described in Section 6. Unlike BPE, WordPiece does not replay the merge rules at tokenization time.

This uses IPython's HTML display and optionally `ipywidgets` for buttons. If `ipywidgets` is unavailable, it displays all five steps as static panels. If buttons are not supported by your notebook viewer, set `WORDPIECE_INTERACTIVE = False` and rerun the cell. No downloads or training of a language model are needed.

In [5]:
from html import escape
from IPython.display import HTML, display

WORDPIECE_INTERACTIVE = True  # Set False for static panels or notebook export.
WORDPIECE_VISUAL_MERGES = 5
VISUAL_UNSEEN_WORD = "bugs"

# Reuse the notebook's algorithm, keeping the visual's state separate from splits.
visual_splits = {word: initial_wordpiece_split(word) for word in word_freqs}
visual_base_vocab = initial_wordpiece_vocab(word_freqs)
visual_steps = []
visual_rules = []

for _ in range(WORDPIECE_VISUAL_MERGES):
    step_scores, step_token_freqs, step_pair_freqs = compute_pair_scores(visual_splits, word_freqs)
    if not step_scores:
        break
    # Same tie breaking as train_wordpiece: score, then pair frequency, then pair.
    winner = select_best_pair(step_scores, step_pair_freqs)
    after, merged_token = merge_wordpiece_pair(winner, visual_splits)
    visual_rules.append((winner, merged_token, step_scores[winner]))
    visual_steps.append({
        "before": visual_splits,
        "scores": step_scores,
        "token_freqs": dict(step_token_freqs),
        "pair_freqs": dict(step_pair_freqs),
        "winner": winner,
        "merged_token": merged_token,
        "after": after,
        "rules": list(visual_rules),
    })
    visual_splits = after


def visual_longest_match(word, vocab, prefix=CONTINUING_SUBWORD_PREFIX):
    """Longest-match (MaxMatch) search, the same procedure as Section 6."""
    vocab = set(vocab)
    tokens, start = [], 0
    while start < len(word):
        for end in range(len(word), start, -1):
            candidate = (prefix if start > 0 else "") + word[start:end]
            if candidate in vocab:
                tokens.append(candidate)
                start = end
                break
        else:
            return [UNK_TOKEN]
    return tokens


def wordpiece_tiles(tokens, pair=None, merged_token=None):
    """Draw token boundaries and highlight the selected merge."""
    selected = set()
    if pair is not None:
        i = 0
        while i < len(tokens) - 1:
            if tuple(tokens[i:i + 2]) == pair:
                selected.update((i, i + 1))
                i += 2
            else:
                i += 1
    tiles = []
    for i, token in enumerate(tokens):
        active = i in selected or token == merged_token
        color = "#fef3c7" if active else "#f1f5f9"
        border = "#a16207" if active else "#64748b"
        tiles.append(
            f'<span style="display:inline-block;margin:3px;padding:5px 9px;'
            f'border:2px solid {border};border-radius:6px;background:{color};'
            f'color:#172033;font-family:monospace">{escape(token)}</span>'
        )
    return "".join(tiles)


def wordpiece_word_rows(splits_to_show, pair=None, merged_token=None):
    return "".join(
        f'<div style="margin:7px 0"><span style="display:inline-block;'
        f'width:85px;font-family:monospace">{escape(word)} ×{word_freqs[word]}</span>'
        f'{wordpiece_tiles(tokens, pair, merged_token)}</div>'
        for word, tokens in splits_to_show.items()
    )


def wordpiece_visual_html(step_index):
    """Render one merge, its score calculation, and the rules learned so far."""
    step = visual_steps[step_index]
    winner = step["winner"]
    new_token = step["merged_token"]
    scores = step["scores"]
    token_freqs = step["token_freqs"]
    pair_freqs = step["pair_freqs"]

    token_counts = " ".join(
        f'<code style="margin-right:8px">{escape(token)}: {count}</code>'
        for token, count in sorted(token_freqs.items())
    )

    bars = []
    # Same order as select_best_pair: the selected pair is listed first.
    ranked = sorted(scores, key=lambda pair: (scores[pair], pair_freqs[pair], pair), reverse=True)
    for pair in ranked:
        first, second = pair
        active = pair == winner
        width = 100 * scores[pair] / scores[ranked[0]]
        color = "#a16207" if active else "#64748b"
        formula = (
            f"{pair_freqs[pair]} / ({token_freqs[first]} × {token_freqs[second]})"
            f" = {scores[pair]:.4f}"
        )
        bars.append(
            f'<div style="margin:8px 0"><div style="font-family:monospace">'
            f'{escape(first)} + {escape(second)}: {escape(formula)}'
            f'{" <b>← selected</b>" if active else ""}</div>'
            f'<div style="background:#e2e8f0;height:10px;border-radius:4px">'
            f'<div style="width:{width}%;height:10px;background:{color};'
            f'border-radius:4px"></div></div></div>'
        )
    tied = [p for p in ranked if abs(scores[p] - scores[winner]) < 1e-12]
    tie_note = (
        f'<p style="font-size:0.9em">{len(tied)} pairs tie on score; the tie is broken by '
        'higher pair frequency, then by reverse alphabetical order of the pair.</p>'
        if len(tied) > 1 else ""
    )

    rules_html = "".join(
        f'<li><code>{escape(a)} + {escape(b)} → {escape(token)}</code>'
        f' (score: {score:.4f})</li>'
        for (a, b), token, score in step["rules"]
    )
    # Tokenization uses the vocabulary, not the merge rules or the scores.
    current_vocab = visual_base_vocab + [token for _, token, _ in step["rules"]]
    unseen = visual_longest_match(VISUAL_UNSEEN_WORD, current_vocab)

    parts = [
        ("① Current token splits", wordpiece_word_rows(step["before"], pair=winner)),
        ("② Pair scores = pair count / (count of first × count of second)",
         f'<div style="margin-bottom:8px">Token counts: {token_counts}</div>'
         + "".join(bars) + tie_note),
        ("③ After merging everywhere",
         wordpiece_word_rows(step["after"], merged_token=new_token)),
    ]
    panels = "".join(
        f'<section style="flex:1;min-width:240px;padding:12px;border:1px solid '
        f'#cbd5e1;border-radius:8px"><h4 style="margin:0 0 12px">{title}</h4>'
        f'{content}</section>' for title, content in parts
    )
    return (
        '<div style="background:white;color:#172033;padding:16px;'
        'border:1px solid #cbd5e1;border-radius:10px;font-family:sans-serif">'
        f'<h3 style="margin-top:0">WordPiece training — merge {step_index + 1}</h3>'
        f'<div style="display:flex;flex-wrap:wrap;gap:12px">{panels}</div>'
        '<p><b>↻ Recount tokens and pairs in the updated splits → rescore → select the next winner.</b></p>'
        f'<p>Vocabulary: {len(visual_base_vocab)} base tokens (including [UNK]) + '
        f'{len(step["rules"])} learned tokens = '
        f'<b>{len(current_vocab)}</b> tokens. Existing vocabulary entries remain available.</p>'
        f'<b>Ordered learned rules</b><ol>{rules_html}</ol>'
        f'<p><b>Longest-match tokenization of unseen word {escape(VISUAL_UNSEEN_WORD)}:</b> '
        f'{wordpiece_tiles(visual_longest_match(VISUAL_UNSEEN_WORD, visual_base_vocab))} → '
        f'{wordpiece_tiles(unseen)}</p>'
        '<p>Training scores pairs and learns merges. Tokenization searches the current '
        'vocabulary for the longest matching piece; it does not replay the merges or use the scores.</p></div>'
    )


try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

if WORDPIECE_INTERACTIVE and widgets is not None:
    visual_position = {"index": 0}
    next_merge_button = widgets.Button(description="Next merge", button_style="primary")
    reset_merge_button = widgets.Button(description="Reset")
    visual_panel = widgets.HTML()

    def show_wordpiece_step():
        visual_panel.value = wordpiece_visual_html(visual_position["index"])
        next_merge_button.disabled = visual_position["index"] == len(visual_steps) - 1

    def advance_wordpiece_step(_):
        visual_position["index"] = min(visual_position["index"] + 1, len(visual_steps) - 1)
        show_wordpiece_step()

    def reset_wordpiece_step(_):
        visual_position["index"] = 0
        show_wordpiece_step()

    next_merge_button.on_click(advance_wordpiece_step)
    reset_merge_button.on_click(reset_wordpiece_step)
    show_wordpiece_step()
    display(widgets.VBox([widgets.HBox([next_merge_button, reset_merge_button]), visual_panel]))
else:
    for step_index in range(len(visual_steps)):
        display(HTML(wordpiece_visual_html(step_index)))

## 5. Training Several WordPiece Merges

Now we can put the steps into a training function.

The function below returns:

- `vocab`: the initial WordPiece vocabulary plus every merged token.
- `merges`: the ordered list of merge rules learned during training.
- `splits`: the final split form of every training word.

The learned merges help us understand training. During tokenization, however, WordPiece usually uses the final vocabulary with a longest-match search rather than replaying every merge rule.

In [6]:
def train_wordpiece(word_freqs, num_merges):
    # Start with WordPiece-style characters.
    # Example: "hug" becomes ["h", "##u", "##g"].
    splits = {word: initial_wordpiece_split(word) for word in word_freqs}

    # The initial vocabulary contains starting characters, continuation characters, and [UNK].
    vocab = initial_wordpiece_vocab(word_freqs)

    # This will store the learned WordPiece rules in order.
    # Each entry will look like: (("##g", "##s"), "##gs", 0.05, 5)
    # meaning: merge the pair into "##gs"; score and pair frequency are stored for inspection.
    merges = []

    # Learn one merge rule per iteration.
    for step in range(num_merges):
        # Compute token frequencies, pair frequencies, and WordPiece scores.
        scores, token_freqs, pair_freqs = compute_pair_scores(splits, word_freqs)

        # If there are no adjacent pairs left, there is nothing more to merge.
        if not scores:
            break

        # Choose the pair with the highest WordPiece score.
        # We also use pair frequency and alphabetical order to make ties deterministic.
        best_pair = select_best_pair(scores, pair_freqs)

        # Merge the selected pair throughout the current training splits.
        splits, merged_token = merge_wordpiece_pair(best_pair, splits)

        # Save the rule for analysis.
        merges.append((best_pair, merged_token, scores[best_pair], pair_freqs[best_pair]))

        # Add the newly created token to the vocabulary.
        vocab.append(merged_token)

        # Print a trace of what the algorithm learned at this step.
        print(
            f"Step {step + 1}: merge {best_pair} -> {merged_token!r}  "
            f"score={scores[best_pair]:.4f}  pair_freq={pair_freqs[best_pair]}"
        )

    return vocab, merges, splits

vocab, merges, final_splits = train_wordpiece(word_freqs, num_merges=5)

print("\nFinal vocabulary:")
print(vocab)

print("\nFinal training-word splits:")
for word, split in final_splits.items():
    print(f"  {word:4s} -> {split}")

Step 1: merge ('##g', '##s') -> '##gs'  score=0.0500  pair_freq=5
Step 2: merge ('p', '##u') -> 'pu'  score=0.0278  pair_freq=17
Step 3: merge ('h', '##u') -> 'hu'  score=0.0526  pair_freq=15
Step 4: merge ('b', '##u') -> 'bu'  score=0.2500  pair_freq=4
Step 5: merge ('hu', '##gs') -> 'hugs'  score=0.0667  pair_freq=5

Final vocabulary:
['##g', '##n', '##s', '##u', '[UNK]', 'b', 'h', 'p', '##gs', 'pu', 'hu', 'bu', 'hugs']

Final training-word splits:
  hug  -> ['hu', '##g']
  pug  -> ['pu', '##g']
  pun  -> ['pu', '##n']
  bun  -> ['bu', '##n']
  hugs -> ['hugs']


### What Did WordPiece Learn?

The tokenizer learned pieces that are useful according to the WordPiece score.

A pair can be selected even if it is not the most frequent pair. This happens when the two parts occur together more strongly than they occur with other neighbors.

WordPiece grows from characters into longer subwords:

```text
characters -> short subwords -> longer subwords -> sometimes whole words
```

Like BPE, WordPiece does not know grammar. It only knows patterns in the training corpus.

## 6. WordPiece Tokenization Algorithm

After training, WordPiece tokenization uses the final vocabulary.

For a new word:

1. Start at the first character.
2. Find the longest vocabulary token that matches the current remaining text.
3. At the beginning of a word, try tokens without `##`.
4. After the first subword, try tokens with the `##` prefix.
5. Move forward and repeat until the whole word is covered.
6. If no valid piece can be found, return `[UNK]` for the whole word.

This longest-match procedure is often called **MaxMatch**.

In [7]:
def tokenize_word(word, vocab, unk_token=UNK_TOKEN, prefix=CONTINUING_SUBWORD_PREFIX):
    """Tokenize one word using WordPiece longest-match tokenization."""
    vocab = set(vocab)
    tokens = []
    start = 0

    # Walk through the word from left to right.
    while start < len(word):
        end = len(word)
        current_token = None

        # Try the longest remaining substring first, then shorten it one character at a time.
        while start < end:
            candidate = word[start:end]
            if start > 0:
                candidate = prefix + candidate

            if candidate in vocab:
                current_token = candidate
                break

            end -= 1

        # If nothing matched, WordPiece returns [UNK] for the whole word.
        if current_token is None:
            return [unk_token]

        tokens.append(current_token)
        start = end

    return tokens

for word in ["hug", "bug", "pug", "unhug", "mug", "thug", "hugs"]:
    print(f"{word:6s} -> {tokenize_word(word, vocab)}")

hug    -> ['hu', '##g']
bug    -> ['bu', '##g']
pug    -> ['pu', '##g']
unhug  -> ['[UNK]']
mug    -> ['[UNK]']
thug   -> ['[UNK]']
hugs   -> ['hugs']


### Tracing the longest-match search

The cell above uses the vocabulary learned after 5 merges:

```text
[UNK], b, h, p, ##g, ##n, ##s, ##u, ##gs, pu, hu, bu, hugs
```

At each position, the search tries the longest remaining substring first and shortens it one character at a time. A piece at the start of the word is looked up as it is; every later piece is looked up with the `##` prefix.

**`hugs`**

| start | candidates tried | match |
| --- | --- | --- |
| 0 | `hugs` ✓ | `hugs` |

Result: `['hugs']`. The whole word is in the vocabulary, so it matches at once.

**`hug`**

| start | candidates tried | match |
| --- | --- | --- |
| 0 | `hug` ✗, `hu` ✓ | `hu` |
| 2 | `##g` ✓ | `##g` |

Result: `['hu', '##g']`.

**`bugs`** (never seen during training)

| start | candidates tried | match |
| --- | --- | --- |
| 0 | `bugs` ✗, `bug` ✗, `bu` ✓ | `bu` |
| 2 | `##gs` ✓ | `##gs` |

Result: `['bu', '##gs']`. An unseen word can still be built from learned pieces.

**`unhug`**

| start | candidates tried | match |
| --- | --- | --- |
| 0 | `unhug` ✗, `unhu` ✗, `unh` ✗, `un` ✗, `u` ✗ | none |

Result: `['[UNK]']`. The letter `u` exists only as the continuation token `##u`, never as a word start, so nothing matches at position 0 and the **whole word** becomes `[UNK]`. The same happens to `mug` and `thug`.

Compare this with the BPE notebook, where only the unknown character is replaced: BPE gives `thug -> ['[UNK]', 'hug']`, while WordPiece gives `thug -> ['[UNK]']`.

**When greedy fails.** The search never backtracks, so it can miss a segmentation that exists. Suppose the vocabulary contains only `a`, `ab`, and `##bc`:

| start | candidates tried | match |
| --- | --- | --- |
| 0 | `abc` ✗, `ab` ✓ | `ab` |
| 2 | `##c` ✗ | none |

Result: `['[UNK]']`, even though `['a', '##bc']` was a valid split. The algorithm committed to the longer piece `ab` and never reconsidered. In practice this is rare, because a large vocabulary contains every character in both its word-start and `##` forms, so some match is always available.

### Unknown Words and Unknown Pieces

WordPiece can represent many unfamiliar words by combining known starting tokens and continuation tokens.

However, this tiny vocabulary is very small. If a word begins with a character that was never seen at the beginning of a training word, the tokenizer cannot start the segmentation and returns `[UNK]`.

For example, `mug` becomes `[UNK]` because `m` is not in the vocabulary as a starting token.

BERT's real vocabulary is much larger and is trained on a large corpus, so this fallback happens less often. It can still happen for text that contains unsupported characters or unusual strings.

## 7. Continuation Tokens: Why `##` Matters

The `##` prefix tells us that a token is attached to the previous subword.

For example, these token sequences mean different things:

```text
["play", "##ing"]  -> "playing"
["play", "ing"]    -> "play ing" or two separate word-like pieces
```

The prefix lets the tokenizer distinguish a piece that begins a word from a piece that continues a word.

This is why the initial WordPiece vocabulary stores the first character of a word normally, but stores later characters with `##`:

```text
token -> t ##o ##k ##e ##n
```

During decoding, the tokenizer can remove `##` and join continuation pieces back to the previous piece.

In [8]:
def decode_wordpieces(tokens, prefix=CONTINUING_SUBWORD_PREFIX):
    """A tiny decoder for display purposes."""
    text = ""
    for token in tokens:
        if token == UNK_TOKEN:
            text += " " + token
        elif token.startswith(prefix):
            text += token.removeprefix(prefix)
        else:
            if text:
                text += " "
            text += token
    return text.strip()

examples = [
    ["hug"],
    ["h", "##u", "##g"],
    ["p", "##u", "##n"],
    ["h", "##u", "##gs"],
]

for tokens in examples:
    print(f"{tokens} -> {decode_wordpieces(tokens)!r}")

['hug'] -> 'hug'
['h', '##u', '##g'] -> 'hug'
['p', '##u', '##n'] -> 'pun'
['h', '##u', '##gs'] -> 'hugs'


## 8. Handling Spaces: BERT-Style Pre-tokenization

WordPiece tokenizers such as BERT usually apply a pre-tokenizer before WordPiece segmentation. A simple version splits words and punctuation into separate pieces.

Unlike GPT-2/RoBERTa-style byte-level BPE, BERT-style WordPiece does not usually show a leading-space marker such as `Ġ`. Spaces are handled by the pre-tokenizer, and continuation inside a word is marked with `##`.

In the code below, punctuation is kept as its own pre-tokenized piece.

In [9]:
def simple_pre_tokenize(text):
    """A tiny BERT-style pre-tokenizer that separates punctuation."""
    pieces = []
    current = ""

    for char in text:
        if char.isspace():
            if current:
                pieces.append(current)
                current = ""
        elif char in ".,!?;:":
            if current:
                pieces.append(current)
                current = ""
            pieces.append(char)
        else:
            current += char

    if current:
        pieces.append(current)

    return pieces

example = "This is tokenization. This is useful!"
print(simple_pre_tokenize(example))

['This', 'is', 'tokenization', '.', 'This', 'is', 'useful', '!']


## 9. Training WordPiece on a Small Sentence Corpus

Now we will train WordPiece on the same small sentence corpus used in the BPE notebook.

The corpus is intentionally tiny so that we can inspect the learned rules. A production tokenizer would be trained on a much larger and more diverse corpus.

In [10]:
sentence_corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will understand how tokenizers generate tokens.",
]

sentence_word_freqs = Counter()
for text in sentence_corpus:
    sentence_word_freqs.update(simple_pre_tokenize(text))

print("Pre-tokenized word frequencies:")
for word, freq in sentence_word_freqs.items():
    print(f"  {word:18s} {freq}")

Pre-tokenized word frequencies:
  This               3
  is                 2
  the                1
  Hugging            1
  Face               1
  Course             1
  .                  4
  chapter            1
  about              1
  tokenization       1
  section            1
  shows              1
  several            1
  tokenizer          1
  algorithms         1
  Hopefully          1
  ,                  1
  you                1
  will               1
  understand         1
  how                1
  tokenizers         1
  generate           1
  tokens             1


In [11]:
sentence_vocab, sentence_merges, sentence_final_splits = train_wordpiece(sentence_word_freqs, num_merges=40)

print("\nVocabulary size:", len(sentence_vocab))
print("First 20 vocabulary items:", sentence_vocab[:20])
print("Last 10 learned tokens:", sentence_vocab[-10:])

Step 1: merge ('a', '##b') -> 'ab'  score=0.5000  pair_freq=1
Step 2: merge ('##f', '##u') -> '##fu'  score=0.2000  pair_freq=1
Step 3: merge ('a', '##l') -> 'al'  score=0.1667  pair_freq=1
Step 4: merge ('al', '##g') -> 'alg'  score=0.2500  pair_freq=1
Step 5: merge ('##l', '##y') -> '##ly'  score=0.2000  pair_freq=1
Step 6: merge ('##l', '##ly') -> '##lly'  score=0.2500  pair_freq=1
Step 7: merge ('##fu', '##lly') -> '##fully'  score=1.0000  pair_freq=1
Step 8: merge ('F', '##a') -> 'Fa'  score=0.1667  pair_freq=1
Step 9: merge ('Fa', '##c') -> 'Fac'  score=0.5000  pair_freq=1
Step 10: merge ('T', '##h') -> 'Th'  score=0.1429  pair_freq=3
Step 11: merge ('c', '##h') -> 'ch'  score=0.2500  pair_freq=1
Step 12: merge ('##h', '##m') -> '##hm'  score=0.3333  pair_freq=1
Step 13: merge ('ch', '##a') -> 'cha'  score=0.2000  pair_freq=1
Step 14: merge ('cha', '##p') -> 'chap'  score=0.5000  pair_freq=1
Step 15: merge ('s', '##h') -> 'sh'  score=0.1667  pair_freq=1
Step 16: merge ('t', '##h'

### Reading the Merge Rules

Each learned merge says that a pair had a high WordPiece score at the moment it was selected.

For example, if the algorithm learns:

```text
("t", "##o") -> "to"
```

then a word beginning with `to` can use the token `to`.

If it later learns:

```text
("to", "##ken") -> "token"
```

then `token` can become one token at the beginning of a word. Inside a longer word, the tokenizer would need the corresponding continuation form, such as `##token`, if that token exists in the vocabulary.

This beginning-versus-continuation distinction is one reason WordPiece vocabularies often contain both normal and `##`-prefixed pieces.

## 10. Tokenizing a New Sentence

To tokenize a full sentence, we:

1. Pre-tokenize it into rough word and punctuation pieces.
2. Apply WordPiece longest-match tokenization to each piece.
3. Flatten the result into one token sequence.

The merge rules are useful for understanding how the vocabulary was trained. The final vocabulary is what drives tokenization.

In [12]:
def tokenize_text(text, vocab):
    pieces = simple_pre_tokenize(text)
    tokens = []

    for piece in pieces:
        tokens.extend(tokenize_word(piece, vocab))

    return tokens

test_sentences = [
    "This is tokenization.",
    "This tokenizer generates tokens.",
    "Unknown emojis are hard 😊",
]

for text in test_sentences:
    print(text)
    print(tokenize_text(text, sentence_vocab))
    print()

This is tokenization.
['Thi', '##s', 'i', '##s', 't', '##o', '##k', '##e', '##n', '##iz', '##at', '##i', '##o', '##n', '.']

This tokenizer generates tokens.
['Thi', '##s', 't', '##o', '##k', '##e', '##n', '##iz', '##e', '##r', 'g', '##e', '##n', '##e', '##rat', '##e', '##s', 't', '##o', '##k', '##e', '##n', '##s', '.']

Unknown emojis are hard 😊
['[UNK]', '[UNK]', 'a', '##r', '##e', 'h', '##a', '##r', '##d', '[UNK]']



The emoji example shows the limitation of this classroom implementation. Since the emoji was never part of the vocabulary, the affected pre-tokenized piece becomes `[UNK]`.

A production tokenizer may use stronger normalization and a much larger vocabulary, but WordPiece still relies on `[UNK]` when it cannot segment a word into known pieces.

## 11. Vocabulary Growth

WordPiece grows the vocabulary one merge at a time.

Suppose the base vocabulary has 30 symbols and we learn 20 merges. The final vocabulary size is:

```text
30 + 20 = 50
```

In practice, vocabulary size is a design choice. A larger vocabulary can make sequences shorter, but it also makes the embedding table and output layer larger.

| Smaller vocabulary | Larger vocabulary |
| --- | --- |
| Longer token sequences | Shorter token sequences |
| Smaller embedding table | Larger embedding table |
| More splitting into small pieces | More whole words and large subwords |
| Better fallback through small pieces | More parameters devoted to token representations |

In [13]:
sentence_base_vocab = initial_wordpiece_vocab(sentence_word_freqs)
base_size = len(sentence_base_vocab)
merge_count = len(sentence_merges)
final_size = len(sentence_vocab)

print("Base vocabulary size:", base_size)
print("Number of learned merges:", merge_count)
print("Final vocabulary size:", final_size)
print("Check: base_size + merge_count =", base_size + merge_count)

Base vocabulary size: 40
Number of learned merges: 40
Final vocabulary size: 80
Check: base_size + merge_count = 80


## 12. BPE vs WordPiece

BPE and WordPiece are both subword tokenization algorithms, but their training choices and inference procedures differ.

| Algorithm | Used by | Main training idea | Common inference idea |
| --- | --- | --- | --- |
| BPE | GPT, GPT-2, RoBERTa, BART | Repeatedly merge the most frequent adjacent pair | Apply learned merge rules in order |
| WordPiece | BERT | Repeatedly merge the highest-scoring adjacent pair | Use longest-match search over the final vocabulary |

The high-level goal is similar: build a vocabulary of useful subword units so the model can handle common words efficiently and still represent rare words through smaller pieces.

The score is the central difference in training:

```text
BPE:       choose the pair with highest frequency
WordPiece: choose the pair with highest frequency(pair) / frequency(parts)
```

So WordPiece may choose a less frequent pair if its parts are strongly associated with each other.

## 13. Classroom Exercise

Try answering these before running more experiments.

1. In the toy corpus, why does `("##g", "##s")` merge before `("##u", "##g")`, even though `("##u", "##g")` has a higher pair frequency?
2. What does the `##` prefix tell us about where a token may appear inside a word?
3. Why does WordPiece use longest-match tokenization after training instead of recounting pair scores for new text?
4. Why might `mug` become `[UNK]` in our tiny tokenizer even though it shares `##u` and `##g` with known words?
5. What is the trade-off between a small vocabulary and a large vocabulary?
6. Train with `num_merges=10`, `20`, and `60`. How does the tokenization of `"This tokenizer generates tokens."` change?

## 14. Key Takeaways

- WordPiece is a **data-driven subword tokenization algorithm**.
- It starts with small units, often characters with `##` marking continuation inside a word.
- During training, it repeatedly merges the adjacent token pair with the highest WordPiece score.
- The score favors pairs whose parts occur together more strongly than they occur separately.
- During tokenization, WordPiece usually applies longest-match search over the final vocabulary.
- If a word cannot be segmented into known pieces, WordPiece returns `[UNK]` for that word.
- Tokenization affects model efficiency because it determines sequence length, vocabulary size, embedding size, and output-layer size.